# Обоснование порога для классификатора utilization

Запуск после загрузки `sample` (колонка `target` — доля использованного лимита).

Графики для спецификации модели:
1. **Чувствительность** — как меняется P(active) при сдвиге порога
2. **CDF** — доля клиентов с утилизацией ≤ X%
3. **Зум у нуля** — где скапливается масса
4. **Корзины** — разбиение на (0], (0;0.1%], …

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# sample = pd.read_parquet("./data/sample.parquet")
TARGET_COL = "target"
y = pd.to_numeric(sample[TARGET_COL], errors="coerce").dropna()
n = len(y)

CANDIDATE_THRESHOLDS = [0, 1e-9, 0.001, 0.01, 0.02, 0.05, 0.10]
HIGHLIGHT_THRESHOLDS = [1e-9, 0.01, 0.05]

In [ ]:
thr_grid = np.unique(np.concatenate([
    [0, 1e-9, 1e-6],
    np.linspace(0, 0.10, 41),
    np.linspace(0.10, 0.50, 9),
]))
thr_sens = []
for thr in thr_grid:
    pos = y > thr
    thr_sens.append({
        "thr": thr,
        "p_active": pos.mean(),
        "n_active": int(pos.sum()),
        "share_dropped_vs_gt0": float((y > 0).mean() - pos.mean()),
    })
thr_df = pd.DataFrame(thr_sens)

show = thr_df[thr_df["thr"].isin(CANDIDATE_THRESHOLDS)].copy()
show["P(active)"] = (100 * show["p_active"]).round(2).astype(str) + "%"
show["отсечено vs порог>0"] = (100 * show["share_dropped_vs_gt0"]).round(2).astype(str) + "%"
display(show[["thr", "P(active)", "n_active", "отсечено vs порог>0"]])

In [ ]:
fig_sens = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "Доля клиентов с утилизацией выше порога",
        "Сколько п.п. «active» теряем при +Δ порога",
    ),
    vertical_spacing=0.12,
)
p_active = thr_df["p_active"].values
fig_sens.add_trace(
    go.Scatter(
        x=thr_df["thr"], y=100 * p_active, mode="lines",
        line=dict(width=2),
        hovertemplate="порог %{x:.4%}<br>P(active) %{y:.2f}%<extra></extra>",
    ),
    row=1, col=1,
)
for thr in HIGHLIGHT_THRESHOLDS:
    fig_sens.add_vline(x=thr, line_dash="dash", line_color="#D62728", row=1, col=1)
    fig_sens.add_annotation(
        x=thr, y=100 * (y > thr).mean(), text=f"{100 * thr:.2g}%",
        showarrow=True, arrowhead=2, ax=25, ay=-20, row=1, col=1,
    )
dp = -np.diff(100 * p_active)
fig_sens.add_trace(
    go.Bar(x=thr_df["thr"].values[1:], y=dp, marker_color="steelblue"),
    row=2, col=1,
)
fig_sens.update_xaxes(title_text="Порог (доля лимита)", row=2, col=1)
fig_sens.update_yaxes(title_text="% active", row=1, col=1)
fig_sens.update_yaxes(title_text="п.п.", row=2, col=1)
fig_sens.update_layout(height=650, width=900, title="Чувствительность бинарного таргета")
fig_sens.show()

In [ ]:
marks = [0, 0.001, 0.01, 0.02, 0.05, 0.10, 0.25, 0.50]
display(pd.DataFrame([
    {"util_порог": f"{100 * t:.1g}%", "доля_ниже": f"{100 * (y <= t).mean():.1f}%",
     "доля_выше": f"{100 * (y > t).mean():.1f}%"}
    for t in marks
]))

x_pct = np.linspace(0, 1.0, 501)
cdf = np.array([(y <= t).mean() for t in x_pct])
fig_cdf = go.Figure(go.Scatter(
    x=100 * x_pct, y=100 * cdf, mode="lines",
    hovertemplate="≤ %{x:.2f}% утилизации<br>% клиентов %{y:.1f}<extra></extra>",
))
for t in HIGHLIGHT_THRESHOLDS:
    fig_cdf.add_vline(x=100 * t, line_dash="dash", line_color="#D62728")
    fig_cdf.add_annotation(
        x=100 * t, y=100 * (y <= t).mean(),
        text=f"≤{100 * t:.2g}%: {100 * (y <= t).mean():.0f}% клиентов",
        showarrow=False, yshift=10,
    )
fig_cdf.update_layout(
    title="CDF: доля клиентов с утилизацией ≤ X%",
    xaxis_title="Утилизация, % от лимита",
    yaxis_title="% клиентов (накопительно)",
    height=450, width=900,
)
fig_cdf.show()

In [ ]:
y_pos = y[y > 0]
fig_zoom = make_subplots(
    rows=1, cols=2,
    subplot_titles=("0–10% (все)", ">0, log-шкала"),
)
hist_all, edges = np.histogram(y.clip(upper=0.10), bins=50, range=(0, 0.10))
centers = (edges[:-1] + edges[1:]) / 2
fig_zoom.add_trace(
    go.Bar(x=100 * centers, y=hist_all, width=100 * np.diff(edges), opacity=0.75),
    row=1, col=1,
)
for thr in HIGHLIGHT_THRESHOLDS:
    if thr <= 0.10:
        fig_zoom.add_vline(x=100 * thr, line_dash="dash", line_color="#D62728", row=1, col=1)

log_bins = np.logspace(np.log10(max(y_pos.min(), 1e-6)), np.log10(y_pos.max()), 60)
counts, edges = np.histogram(y_pos, bins=log_bins)
fig_zoom.add_trace(
    go.Bar(x=100 * (edges[:-1] + edges[1:]) / 2, y=counts, width=100 * np.diff(edges) * 0.9, opacity=0.75),
    row=1, col=2,
)
for thr in HIGHLIGHT_THRESHOLDS:
    fig_zoom.add_vline(x=100 * thr, line_dash="dash", line_color="#D62728", row=1, col=1)
fig_zoom.update_xaxes(title_text="% утилизации", row=1, col=1)
fig_zoom.update_xaxes(title_text="% утилизации", type="log", row=1, col=2)
fig_zoom.update_layout(height=420, width=1000, title="Распределение у нуля")
fig_zoom.show()

In [ ]:
bins_def = [
    ("Ровно 0", y <= 0),
    ("(0; 0.1%]", (y > 0) & (y <= 0.001)),
    ("(0.1%; 1%]", (y > 0.001) & (y <= 0.01)),
    ("(1%; 2%]", (y > 0.01) & (y <= 0.02)),
    ("(2%; 5%]", (y > 0.02) & (y <= 0.05)),
    ("(5%; 10%]", (y > 0.05) & (y <= 0.10)),
    ("> 10%", y > 0.10),
]
bucket_df = pd.DataFrame([
    {"bucket": label, "n": int(mask.sum()), "share": f"{100 * mask.mean():.2f}%",
     "mean_in_bucket": f"{100 * y[mask].mean():.2f}%" if mask.any() else "—"}
    for label, mask in bins_def
])
display(bucket_df)

fig_bucket = go.Figure(go.Bar(
    x=bucket_df["bucket"], y=bucket_df["n"],
    text=bucket_df["share"], textposition="outside",
))
fig_bucket.update_layout(title="Корзины утилизации", yaxis_title="N", height=450, width=950)
fig_bucket.show()

print(
    f"N={n:,} | median={100*y.median():.2f}% | mean={100*y.mean():.2f}% | "
    f"P(y=0)={100*(y<=0).mean():.1f}% | P(y>0)={100*(y>0).mean():.1f}%"
)